In [4]:
import json
 
# --- Data ---
import pandas as pd
import numpy as np
 
# --- PyTorch + HuggingFace ---
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer
 
# --- Vector Database ---
import chromadb

In [5]:
BERT_MODEL_DIR = "./fashion-bert"       # where you saved trained BERT
CHROMA_DIR     = "./chromadb_store"     # where catalog pipeline saved ChromaDB
MAX_LEN        = 64
TOP_K          = 5                      # number of products to return
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
 
# Must match exactly what you trained with
import json

with open("./fashion-bert/label_maps.json", "r") as f:
    LABEL_MAPS = json.load(f)

REVERSE_MAPS = {cat: {v: k for k, v in m.items()} for cat, m in LABEL_MAPS.items()}

In [6]:
WEIGHTS = {
    "occasion":   0.35,   # most important — right occasion is critical
    "formality":  0.25,   # second most important
    "constraint": 0.25,   # key differentiator (understated vs bold)
    "color_tone": 0.35,   # least important
}

In [7]:
from transformers import BertModel, BertTokenizer
import torch
import torch.nn as nn
import json


In [8]:
from transformers import BertModel, BertTokenizer
import torch
import torch.nn as nn
import json

# --- Config ---
BERT_MODEL_DIR = "./fashion-bert"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Label Maps ---
with open("./fashion-bert/label_maps.json", "r") as f:
    LABEL_MAPS = json.load(f)

REVERSE_MAPS = {cat: {v: k for k, v in m.items()} for cat, m in LABEL_MAPS.items()}

# --- Model ---
class FashionIntentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        hidden = self.bert.config.hidden_size
        self.occasion_head   = nn.Linear(hidden, len(LABEL_MAPS["occasion"]))
        self.formality_head  = nn.Linear(hidden, len(LABEL_MAPS["formality"]))
        self.constraint_head = nn.Linear(hidden, len(LABEL_MAPS["constraint"]))
        self.color_head      = nn.Linear(hidden, len(LABEL_MAPS["color_tone"]))
        self.dropout         = nn.Dropout(0.3)

    def forward(self, input_ids, attention_mask):
        output    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = self.dropout(output.last_hidden_state[:, 0, :])
        return {
            "occasion":   self.occasion_head(cls_token),
            "formality":  self.formality_head(cls_token),
            "constraint": self.constraint_head(cls_token),
            "color":      self.color_head(cls_token),
        }

# --- Load Function ---
def load_bert():
    print("Loading BERT model...")
    tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_DIR)
    model     = FashionIntentModel().to(DEVICE)
    model.load_state_dict(
        torch.load(f"{BERT_MODEL_DIR}/model.pt", map_location=DEVICE)
    )
    model.eval()
    print("BERT loaded.")
    return model, tokenizer

# --- Run ---
bert_model, tokenizer = load_bert()

Loading BERT model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5764.43it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT loaded.


In [9]:
def extract_intent(prompt, model, tokenizer):
    """
    Takes a raw user prompt.
    Returns structured intent dict with predicted labels + confidence scores.
    """
    tokens = tokenizer(
        prompt,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
 
    input_ids      = tokens["input_ids"].to(DEVICE)
    attention_mask = tokens["attention_mask"].to(DEVICE)
 
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
 
    # Get predicted label + confidence scores for each category
    occasion_scores   = outputs["occasion"].softmax(dim=1).cpu().tolist()[0]
    formality_scores  = outputs["formality"].softmax(dim=1).cpu().tolist()[0]
    constraint_scores = outputs["constraint"].softmax(dim=1).cpu().tolist()[0]
    color_scores      = outputs["color"].softmax(dim=1).cpu().tolist()[0]
 
    intent = {
        "occasion":   REVERSE_MAPS["occasion"][np.argmax(occasion_scores)],
        "formality":  REVERSE_MAPS["formality"][np.argmax(formality_scores)],
        "constraint": REVERSE_MAPS["constraint"][np.argmax(constraint_scores)],
        "color_tone": REVERSE_MAPS["color_tone"][np.argmax(color_scores)],
 
        # Full probability distributions — used in scoring
        "scores": {
            "occasion":   occasion_scores,
            "formality":  formality_scores,
            "constraint": constraint_scores,
            "color_tone": color_scores,
        }
    }
 
    return intent

In [10]:
def query_chromadb(intent, top_k=TOP_K * 3):
    """
    Queries ChromaDB using the predicted occasion as the main search term.
    Returns more results than needed so scoring can re-rank them.
    """
    client     = chromadb.PersistentClient(path=CHROMA_DIR)
    collection = client.get_collection("fashion_products")
 
    # Use occasion + constraint as the query text for better retrieval
    query_text = f"{intent['occasion']} {intent['formality']} {intent['constraint']} outfit"
 
    results = collection.query(
        query_texts=[query_text],
        n_results=min(top_k, collection.count()),
    )
 
    products = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        products.append({
            "title":            meta.get("title", doc),
            "price":            meta.get("price", ""),
            "category":         meta.get("category", ""),
            "local_image_path": meta.get("local_image_path", ""),
            "occasion_scores":  json.loads(meta.get("occasion_scores",  "{}")),
            "boldness_scores":  json.loads(meta.get("boldness_scores",  "{}")),
            "color_scores":     json.loads(meta.get("color_scores",     "{}")),
            "formality_scores": json.loads(meta.get("formality_scores", "{}")),
        })
 
    return products

In [11]:
# score and rank

In [12]:
def score_product(product, intent):
    """
    Scores a single product against the extracted user intent.
    Higher score = better match.
 
    For each category:
    - Find the CLIP score of the product for the predicted label
    - Multiply by that category's weight
    - Apply constraint penalty if needed
    """
 
    score         = 0.0
    score_details = {}
 
    # --- Occasion Score ---
    predicted_occasion = intent["occasion"]
    occasion_label     = f"{predicted_occasion} outfit"
 
    # Find closest matching CLIP label
    occ_scores  = product["occasion_scores"]
    occ_match   = max(
        (v for k, v in occ_scores.items() if predicted_occasion in k),
        default=0.0
    )
    occasion_contribution  = occ_match * WEIGHTS["occasion"]
    score                 += occasion_contribution
    score_details["occasion"] = round(occasion_contribution, 3)
 
    # --- Formality Score ---
    predicted_formality = intent["formality"]
    form_scores         = product["formality_scores"]
    form_match          = max(
        (v for k, v in form_scores.items() if predicted_formality in k),
        default=0.0
    )
    formality_contribution  = form_match * WEIGHTS["formality"]
    score                  += formality_contribution
    score_details["formality"] = round(formality_contribution, 3)
 
    # --- Constraint Score (with penalty logic) ---
    predicted_constraint = intent["constraint"]
    bold_scores          = product["boldness_scores"]
 
    if predicted_constraint == "understated":
        # User wants subtle — reward low boldness score
        subtle_score = max(
            (v for k, v in bold_scores.items() if "subtle" in k or "understated" in k),
            default=0.0
        )
        constraint_contribution = subtle_score * WEIGHTS["constraint"]
 
    elif predicted_constraint == "bold":
        # User wants bold — reward high boldness score
        bold_score = max(
            (v for k, v in bold_scores.items() if "bold" in k or "loud" in k),
            default=0.0
        )
        constraint_contribution = bold_score * WEIGHTS["constraint"]
 
    else:
        # no_constraint — neutral, give partial score
        constraint_contribution = 0.5 * WEIGHTS["constraint"]
 
    score                     += constraint_contribution
    score_details["constraint"] = round(constraint_contribution, 3)
 
    # --- Color Score ---
    predicted_color = intent["color_tone"]
    col_scores      = product["color_scores"]
    col_match       = max(
        (v for k, v in col_scores.items() if predicted_color in k),
        default=0.0
    )
    color_contribution  = col_match * WEIGHTS["color_tone"]
    score              += color_contribution
    score_details["color_tone"] = round(color_contribution, 3)
 
    return round(score, 4), score_details
 
 
def rank_products(products, intent, top_k=TOP_K):
    """
    Scores all retrieved products and returns top_k ranked results.
    """
    scored = []
 
    for product in products:
        score, details = score_product(product, intent)
        scored.append({
            **product,
            "match_score":    score,
            "score_details":  details,
        })
 
    # Sort by score descending
    scored.sort(key=lambda x: x["match_score"], reverse=True)
 
    return scored[:top_k]

In [13]:
# 5. DISPLAY RESULTS
# ============================================================
 
def display_results(prompt, intent, ranked_products):
    """
    Prints results in a clean readable format.
    """
    print("\n" + "=" * 60)
    print(f"Prompt   : {prompt}")
    print(f"Occasion : {intent['occasion']}")
    print(f"Formality: {intent['formality']}")
    print(f"Constraint:{intent['constraint']}")
    print(f"Color    : {intent['color_tone']}")
    print("=" * 60)
 
    print(f"\nTop {len(ranked_products)} Recommendations:\n")
 
    for i, product in enumerate(ranked_products):
        print(f"{i+1}. {product['title']}")
        print(f"   Price      : {product['price']}")
        print(f"   Category   : {product['category']}")
        print(f"   Match Score: {product['match_score']}")
        print(f"   Why matched: occasion={product['score_details']['occasion']} | "
              f"formality={product['score_details']['formality']} | "
              f"constraint={product['score_details']['constraint']} | "
              f"color={product['score_details']['color_tone']}")
        print()
 

In [14]:
# Item type keyword filter
ITEM_KEYWORDS = {
    "jeans":    ["jeans", "denim"],
    "dress":    ["dress", "maxi", "mini", "bodycon", "gown"],
    "top":      ["top", "blouse"],
    "tshirt":   ["t-shirt", "tshirt", "graphic tee", "crew neck"],
    "co-ord":   ["co-ord", "coord", "set"],
    "jumpsuit": ["jumpsuit", "playsuit"],
    "skirt":    ["skirt"],
    "pants":    ["pants", "trousers", "trackpants", "joggers"],
    "shorts":   ["shorts"],
}

def filter_by_item_type(prompt, candidates):
    prompt_lower = prompt.lower()

    # Find requested item type from prompt
    requested_keywords = None
    for item_type, keywords in ITEM_KEYWORDS.items():
        if any(kw in prompt_lower for kw in keywords):
            requested_keywords = keywords
            break

    # No specific item mentioned — return all
    if not requested_keywords:
        return candidates

    # Filter candidates by title
    filtered = [
        p for p in candidates
        if any(kw in p["title"].lower() for kw in requested_keywords)
    ]

    print(f"Item filter: {len(filtered)}/{len(candidates)} products match")

    # Fallback to all if nothing matched
    return filtered if filtered else candidates

In [15]:
def recommend(prompt, top_k=TOP_K):
    intent     = extract_intent(prompt, bert_model, tokenizer)
    candidates = query_chromadb(intent, top_k=top_k * 3)
    if not candidates:
        return []

    # Filter by item type ← new line
    candidates = filter_by_item_type(prompt, candidates)

    ranked = rank_products(candidates, intent, top_k=top_k)
    display_results(prompt, intent, ranked)
    return ranked

In [16]:
# 7. MAIN
# ============================================================
 
if __name__ == "__main__":
 
    # Load models once
    bert_model, tokenizer = load_bert()
 
    # Test prompts
    test_prompts = [
        "going to my friend's birthday, want to look nice but not steal her attention",
        "job interview tomorrow, need to look professional",
        "my ex will be there, i need to look amazing",
        "beach vacation with my girls, something fun and colorful",
        "office party, need something that works for day and night",
    ]
 
    for prompt in test_prompts:
        results = recommend(prompt, bert_model, tokenizer)
 
    # Interactive mode — type your own prompts
    print("\n" + "=" * 60)
    print("Interactive Mode — type a prompt, press Enter")
    print("Type 'quit' to exit")
    print("=" * 60 + "\n")
 
    while True:
        user_input = input("Your prompt: ").strip()
        if user_input.lower() == "quit":
            break
        if user_input:
            recommend(user_input, bert_model, tokenizer)
 

Loading BERT model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14322.89it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT loaded.


TypeError: recommend() takes from 1 to 2 positional arguments but 3 were given

In [ ]:
import inspect
print(inspect.getsource(FashionIntentModel))

OSError: source code not available